# LeetCode 589 — N-ary Tree Preorder Traversal bằng Python

**Link đề:** https://leetcode.com/problems/n-ary-tree-preorder-traversal/
**Độ khó:** Easy · **Chủ đề:** Stack, Tree, Depth-First Search

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children
"""

class Solution:
    def preorder(self, root: 'Node') -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu **cây N-ary** (mỗi node có thể có nhiều con, lưu trong list `children`) và cách LeetCode ghi cây này thành list có `null` làm dấu ngăn nhóm con.
2. Hiểu **duyệt tiền thứ tự (preorder)**: thăm node trước, rồi lần lượt duyệt từng cây con từ trái sang phải.
3. Viết lời giải **đệ quy** (DFS) và thấy rõ call stack; biết giới hạn đệ quy của Python.
4. Trả lời câu Follow-up: viết lời giải **lặp (iterative)** bằng **stack**, hiểu vì sao phải đẩy con theo thứ tự **ngược**.
5. Trace từng bước, chứng minh bằng invariant, phân tích Time/Space, tự làm 6 bài tập có đáp án cuối notebook.

## 1. Đề bài và ví dụ

Cho `root` của một **cây N-ary**, trả về list các giá trị node theo thứ tự **preorder**.

Ví dụ 1: `root = [1,null,3,2,4,null,5,6]` → `[1,3,5,6,2,4]`

```text
            1
         /  |  \
        3   2   4
       / \
      5   6
```

Preorder = "gặp node nào ghi node đó trước, rồi mới đi vào các con từ trái qua phải":
`1` → vào con đầu `3` → vào con của 3 là `5`, `6` → quay lên, sang `2` → sang `4`.

Ví dụ 2: `root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]`
→ `[1,2,3,6,7,11,14,4,8,12,5,9,13,10]`

**Constraints:**
- Số node trong `[0, 10^4]` (có thể **rỗng** → trả `[]`).
- `0 <= Node.val <= 10^4`.
- Chiều cao cây `<= 1000`.

**Follow up:** đệ quy thì dễ, bạn có làm được bằng vòng lặp (iterative) không?

## 2. Nền tảng: cây N-ary, `class Node` và cách LeetCode ghi cây

### 2.1. Node có list con
Khác cây nhị phân (chỉ có `left`, `right`), node N-ary có `children` là một **list** các node con, theo thứ tự từ trái sang phải.

- **root**: node trên cùng.
- **leaf (lá)**: node có `children` rỗng (`[]`) hoặc `None`.
- **subtree (cây con)**: một node cùng toàn bộ con cháu của nó.

Ta tự định nghĩa `Node` giống LeetCode để notebook chạy độc lập. Lưu ý: ta dùng `children or []` để mọi node luôn có list (tránh lỗi khi `children=None`).

In [1]:
from collections import deque
from typing import List, Optional
import random, sys


class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children if children is not None else []

    def __repr__(self):
        return f"Node({self.val})"


# Dựng tay cây ở ví dụ 1
n5, n6 = Node(5), Node(6)
n3 = Node(3, [n5, n6])
root_demo = Node(1, [n3, Node(2), Node(4)])

print(root_demo, "có các con:", root_demo.children)
print(n3, "có các con:", n3.children)
print("Node 2 là lá?", root_demo.children[1].children == [])

Node(1) có các con: [Node(3), Node(2), Node(4)]
Node(3) có các con: [Node(5), Node(6)]
Node 2 là lá? True


### 2.2. Cách LeetCode ghi cây N-ary thành list

LeetCode ghi theo **tầng (level order)**, và **mỗi nhóm con** của một node được ngăn bằng `null` (`None` trong Python):

```text
[1, null, 3, 2, 4, null, 5, 6]
 ^   ^    \_____/   ^   \__/
root ngăn  con của 1 ngăn con của 3   (con của 2, 4 rỗng → không cần ghi tiếp)
```

Quy tắc đọc: lấy node theo thứ tự xuất hiện (dùng **queue** – hàng đợi), mỗi lần gặp `null` thì chuyển sang "nhóm con của node kế tiếp trong hàng đợi".

### 2.3. `deque` – hàng đợi hai đầu
`deque` (trong `collections`) cho phép `append` (thêm cuối) và `popleft` (lấy đầu) đều nhanh `O(1)`. Đây là công cụ chuẩn khi cần "ai vào trước ra trước" (FIFO).

In [2]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("Hàng đợi:", q)
print("popleft ->", q.popleft())   # A ra trước
print("Còn lại:", q)

Hàng đợi: deque(['A', 'B', 'C'])
popleft -> A
Còn lại: deque(['B', 'C'])


Giờ viết hàm dựng cây `build_nary(values)` và hàm in cây `show_nary(root)` (in thụt lề theo độ sâu). Hai hàm này chỉ để học/test, **không** nộp lên LeetCode.

In [3]:
def build_nary(values):
    """Dựng cây N-ary từ list kiểu LeetCode (None là dấu ngăn nhóm con)."""
    if not values:
        return None
    root = Node(values[0])
    queue = deque([root])
    i = 1                      # values[1] là dấu ngăn sau root
    n = len(values)
    while queue and i < n:
        i += 1                 # bỏ qua dấu None mở đầu nhóm con
        parent = queue.popleft()
        while i < n and values[i] is not None:
            child = Node(values[i])
            parent.children.append(child)
            queue.append(child)
            i += 1
    return root


def show_nary(root, depth=0):
    """In cây thụt lề: mỗi tầng sâu hơn thụt thêm 4 dấu cách."""
    if root is None:
        print("(cây rỗng)")
        return
    print("    " * depth + str(root.val))
    for child in root.children:
        show_nary(child, depth + 1)


EX1 = [1, None, 3, 2, 4, None, 5, 6]
EX2 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]

show_nary(build_nary(EX1))
print("-----")
show_nary(build_nary(EX2))

1
    3
        5
        6
    2
    4
-----
1
    2
    3
        6
        7
            11
                14
    4
        8
            12
    5
        9
            13
        10


## 3. Preorder là gì? Tính tay trước khi code

**Preorder (tiền thứ tự)** – định nghĩa đệ quy:

```text
preorder(node) = [node.val] + preorder(con_1) + preorder(con_2) + ... + preorder(con_k)
```

Chữ "pre" (trước) nghĩa là **node cha được ghi TRƯỚC** các con. Đối lập: **postorder** (bài 590) ghi cha **SAU** các con.

Tính tay ví dụ 1:

```text
preorder(1) = [1] + preorder(3) + preorder(2) + preorder(4)
preorder(3) = [3] + preorder(5) + preorder(6) = [3, 5, 6]
preorder(2) = [2],  preorder(4) = [4]
=> [1, 3, 5, 6, 2, 4]
```

Câu hỏi tự kiểm tra: với ví dụ 2, sau `3` sẽ là gì? (Đáp: `6`, vì `6` là con đầu của `3`; rồi `7`, rồi các con cháu của 7 là `11`, `14`.)

## 4. Cách 1 — Đệ quy với hàm helper `dfs`

### 4.1. Đệ quy là gì?
Một hàm **gọi lại chính nó** trên bài toán nhỏ hơn. Luôn cần:
1. **Base case** (điểm dừng): ở đây là `node is None` → không làm gì.
2. **Bước đệ quy**: xử lý node hiện tại, rồi gọi cho từng con.

Ví dụ mini: đếm ngược bằng đệ quy.

In [4]:
def countdown(k):
    if k == 0:            # base case
        print("Hết!")
        return
    print(k)
    countdown(k - 1)      # gọi lại với bài toán nhỏ hơn

countdown(3)

3
2
1
Hết!


### 4.2. Áp dụng cho preorder
Ý tưởng: dùng một list `result` chung; hàm `dfs(node)`:
- nếu `node` là `None` → return;
- `result.append(node.val)` **trước** (đây là chữ "pre");
- `for child in node.children: dfs(child)`.

Hàm `dfs` lồng bên trong hàm chính nên "nhìn thấy" được biến `result` (closure). Ta chỉ **gọi phương thức** `result.append(...)`, không gán lại `result = ...`, nên không cần `nonlocal`.

In [5]:
def preorder_recursive(root):
    result = []

    def dfs(node):
        if node is None:
            return
        result.append(node.val)      # ghi node TRƯỚC
        for child in node.children:  # rồi duyệt từng con, trái -> phải
            dfs(child)

    dfs(root)
    return result


print(preorder_recursive(build_nary(EX1)))   # [1, 3, 5, 6, 2, 4]
print(preorder_recursive(build_nary(EX2)))
print(preorder_recursive(None))              # []

[1, 3, 5, 6, 2, 4]
[1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]
[]


### 4.3. Nhìn call stack bằng trace
Mỗi lần gọi `dfs(child)`, Python đặt một "khung gọi" (frame) lên **call stack**; khi hàm con xong, frame được gỡ ra và ta quay lại vòng `for` của cha. Ta in thụt lề theo độ sâu để thấy điều đó.

In [6]:
def trace_preorder_recursive(root):
    result = []

    def dfs(node, depth):
        pad = "  " * depth
        print(f"{pad}vào dfs({node.val}) -> ghi {node.val}")
        result.append(node.val)
        for child in node.children:
            dfs(child, depth + 1)
        print(f"{pad}rời dfs({node.val}), result = {result}")

    if root:
        dfs(root, 0)
    return result

trace_preorder_recursive(build_nary(EX1))

vào dfs(1) -> ghi 1
  vào dfs(3) -> ghi 3
    vào dfs(5) -> ghi 5
    rời dfs(5), result = [1, 3, 5]
    vào dfs(6) -> ghi 6
    rời dfs(6), result = [1, 3, 5, 6]
  rời dfs(3), result = [1, 3, 5, 6]
  vào dfs(2) -> ghi 2
  rời dfs(2), result = [1, 3, 5, 6, 2]
  vào dfs(4) -> ghi 4
  rời dfs(4), result = [1, 3, 5, 6, 2, 4]
rời dfs(1), result = [1, 3, 5, 6, 2, 4]


[1, 3, 5, 6, 2, 4]

### 4.4. Biến thể đệ quy "ghép list" (dễ viết nhưng tốn hơn)

Có thể viết sát định nghĩa toán học: `[node.val] + preorder(con_1) + ...`. Cách này rất "đẹp" nhưng **mỗi phép `+` tạo list mới và copy phần tử**. Với cây dạng chuỗi dài (mỗi node một con), tổng số lần copy có thể tới `1 + 2 + ... + n = O(n²)`.

In [7]:
def preorder_concat(root):
    if root is None:
        return []
    out = [root.val]
    for child in root.children:
        out += preorder_concat(child)    # copy toàn bộ list con vào out
    return out

print(preorder_concat(build_nary(EX1)))
print(preorder_concat(build_nary(EX2)) == preorder_recursive(build_nary(EX2)))

[1, 3, 5, 6, 2, 4]
True


### 4.5. Code nộp LeetCode — Cách 1 (đệ quy)

LeetCode đã có sẵn `Node`, `List`; ta chỉ nộp `class Solution`.

In [8]:
class Solution:
    def preorder(self, root: 'Node') -> List[int]:
        result = []

        def dfs(node):
            if node is None:
                return
            result.append(node.val)
            for child in node.children:
                dfs(child)

        dfs(root)
        return result


print(Solution().preorder(build_nary(EX1)))

[1, 3, 5, 6, 2, 4]


## 5. Nút thắt của đệ quy: giới hạn độ sâu

Python giới hạn số frame trên call stack (mặc định khoảng `1000`, xem bằng `sys.getrecursionlimit()`). Đề nói chiều cao cây tối đa `1000` – sát nút! Trên máy cá nhân, cây dạng "chuỗi" sâu vài nghìn sẽ gây `RecursionError`.

(LeetCode thường nâng giới hạn nên cách đệ quy vẫn Accepted, nhưng đây chính là lý do Follow-up hỏi cách **lặp**.)

In [9]:
def make_chain(length):
    """Cây 'chuỗi': 0 -> 1 -> 2 -> ... (mỗi node đúng 1 con)."""
    root = Node(0)
    cur = root
    for v in range(1, length):
        nxt = Node(v)
        cur.children.append(nxt)
        cur = nxt
    return root

print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
deep = make_chain(20000)
try:
    preorder_recursive(deep)
    print("Đệ quy chạy được")
except RecursionError as e:
    print("Đệ quy lỗi:", type(e).__name__)

Giới hạn đệ quy hiện tại: 1000
Đệ quy lỗi: RecursionError


## 6. Insight: tự quản lý "call stack" bằng một list làm **stack**

### 6.1. Stack (ngăn xếp) – LIFO
**Stack** = vào sau ra trước (Last In, First Out), như chồng đĩa. Trong Python dùng `list`:
- `stack.append(x)`: đặt lên đỉnh – `O(1)`.
- `stack.pop()`: lấy phần tử ở đỉnh (cuối list) – `O(1)`.

In [10]:
stack = []
stack.append(1)
stack.append(2)
stack.append(3)
print("stack:", stack)
print("pop ->", stack.pop())   # 3: vào sau cùng, ra đầu tiên
print("stack:", stack)

stack: [1, 2, 3]
pop -> 3
stack: [1, 2]


### 6.2. Vì sao phải đẩy con theo thứ tự **ngược**?

Preorder cần thăm con **trái nhất trước**. Nhưng stack lấy ra phần tử **vào sau cùng**. Vậy nếu muốn `3` ra trước `2` và `4`, ta phải đẩy `4`, rồi `2`, rồi `3` (đẩy ngược).

`reversed(lst)` duyệt list từ cuối về đầu **mà không tạo list mới**; `lst[::-1]` thì tạo bản sao đảo ngược.

In [11]:
children = ["3", "2", "4"]
print("reversed:", list(reversed(children)))
print("slicing [::-1]:", children[::-1])

stack = []
for c in reversed(children):
    stack.append(c)
print("stack sau khi đẩy ngược:", stack)
print("pop lần lượt:", stack.pop(), stack.pop(), stack.pop())   # 3 2 4 đúng thứ tự trái->phải

reversed: ['4', '2', '3']
slicing [::-1]: ['4', '2', '3']
stack sau khi đẩy ngược: ['4', '2', '3']
pop lần lượt: 3 2 4


## 7. Cách 2 — Lặp bằng stack: giảng từng đoạn rồi ghép

**Bước 1 — trường hợp rỗng:** `if root is None: return []`.

**Bước 2 — khởi tạo:** `stack = [root]`, `result = []`.

**Bước 3 — vòng lặp:** khi stack còn phần tử:
- `node = stack.pop()` → node này là node **kế tiếp trong preorder**, ghi ngay `result.append(node.val)`.
- Đẩy các con theo thứ tự ngược: `for child in reversed(node.children): stack.append(child)`.

Ghép lại:

In [12]:
def preorder_iterative(root):
    if root is None:
        return []
    stack = [root]
    result = []
    while stack:
        node = stack.pop()                    # lấy node ở đỉnh
        result.append(node.val)               # ghi ngay (pre)
        for child in reversed(node.children): # con trái nhất vào sau cùng
            stack.append(child)               #   -> sẽ ra đầu tiên
    return result


print(preorder_iterative(build_nary(EX1)))
print(preorder_iterative(build_nary(EX2)))
print(preorder_iterative(None))
print("Cây chuỗi 20000 node, 5 giá trị đầu:", preorder_iterative(deep)[:5])

[1, 3, 5, 6, 2, 4]
[1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]
[]
Cây chuỗi 20000 node, 5 giá trị đầu: [0, 1, 2, 3, 4]


Mẹo gọn: `stack.extend(reversed(node.children))` tương đương vòng `for ... append`. `extend` thêm **từng phần tử** của một iterable vào cuối list.

In [13]:
s = [10]
s.extend(reversed([3, 2, 4]))
print(s)        # [10, 4, 2, 3]

[10, 4, 2, 3]


## 8. TRACE từng bước cách stack

Bảng cho ví dụ 1 (`stack` ghi từ đáy → đỉnh, đỉnh là phần tử cuối):

| Bước | stack trước pop | pop | result sau khi ghi | Đẩy (ngược) | stack sau |
|---:|---|:---:|---|---|---|
| 1 | `[1]` | 1 | `[1]` | 4, 2, 3 | `[4, 2, 3]` |
| 2 | `[4, 2, 3]` | 3 | `[1, 3]` | 6, 5 | `[4, 2, 6, 5]` |
| 3 | `[4, 2, 6, 5]` | 5 | `[1, 3, 5]` | – | `[4, 2, 6]` |
| 4 | `[4, 2, 6]` | 6 | `[1, 3, 5, 6]` | – | `[4, 2]` |
| 5 | `[4, 2]` | 2 | `[1, 3, 5, 6, 2]` | – | `[4]` |
| 6 | `[4]` | 4 | `[1, 3, 5, 6, 2, 4]` | – | `[]` |

Ý nghĩa cột "stack": đây là danh sách **những cây con còn đang chờ duyệt**; phần tử ở đỉnh luôn là cây con phải duyệt kế tiếp.

In [14]:
def trace_preorder_stack(root):
    if root is None:
        print("Cây rỗng -> []")
        return []
    stack = [root]
    result = []
    step = 1
    while stack:
        before = [x.val for x in stack]
        node = stack.pop()
        result.append(node.val)
        pushed = [c.val for c in reversed(node.children)]
        stack.extend(reversed(node.children))
        print(f"Bước {step}: stack trước={before} | pop {node.val} | result={result} "
              f"| đẩy {pushed} | stack sau={[x.val for x in stack]}")
        step += 1
    return result

trace_preorder_stack(build_nary(EX1))

Bước 1: stack trước=[1] | pop 1 | result=[1] | đẩy [4, 2, 3] | stack sau=[4, 2, 3]
Bước 2: stack trước=[4, 2, 3] | pop 3 | result=[1, 3] | đẩy [6, 5] | stack sau=[4, 2, 6, 5]
Bước 3: stack trước=[4, 2, 6, 5] | pop 5 | result=[1, 3, 5] | đẩy [] | stack sau=[4, 2, 6]
Bước 4: stack trước=[4, 2, 6] | pop 6 | result=[1, 3, 5, 6] | đẩy [] | stack sau=[4, 2]
Bước 5: stack trước=[4, 2] | pop 2 | result=[1, 3, 5, 6, 2] | đẩy [] | stack sau=[4]
Bước 6: stack trước=[4] | pop 4 | result=[1, 3, 5, 6, 2, 4] | đẩy [] | stack sau=[]


[1, 3, 5, 6, 2, 4]

Trace thêm một cây "khó" hơn – ví dụ 2 – để thấy stack có lúc chứa nhiều cây con đang chờ (các anh em bên phải chưa tới lượt):

In [15]:
trace_preorder_stack(build_nary(EX2))

Bước 1: stack trước=[1] | pop 1 | result=[1] | đẩy [5, 4, 3, 2] | stack sau=[5, 4, 3, 2]
Bước 2: stack trước=[5, 4, 3, 2] | pop 2 | result=[1, 2] | đẩy [] | stack sau=[5, 4, 3]
Bước 3: stack trước=[5, 4, 3] | pop 3 | result=[1, 2, 3] | đẩy [7, 6] | stack sau=[5, 4, 7, 6]
Bước 4: stack trước=[5, 4, 7, 6] | pop 6 | result=[1, 2, 3, 6] | đẩy [] | stack sau=[5, 4, 7]
Bước 5: stack trước=[5, 4, 7] | pop 7 | result=[1, 2, 3, 6, 7] | đẩy [11] | stack sau=[5, 4, 11]
Bước 6: stack trước=[5, 4, 11] | pop 11 | result=[1, 2, 3, 6, 7, 11] | đẩy [14] | stack sau=[5, 4, 14]
Bước 7: stack trước=[5, 4, 14] | pop 14 | result=[1, 2, 3, 6, 7, 11, 14] | đẩy [] | stack sau=[5, 4]
Bước 8: stack trước=[5, 4] | pop 4 | result=[1, 2, 3, 6, 7, 11, 14, 4] | đẩy [8] | stack sau=[5, 8]
Bước 9: stack trước=[5, 8] | pop 8 | result=[1, 2, 3, 6, 7, 11, 14, 4, 8] | đẩy [12] | stack sau=[5, 12]
Bước 10: stack trước=[5, 12] | pop 12 | result=[1, 2, 3, 6, 7, 11, 14, 4, 8, 12] | đẩy [] | stack sau=[5]
Bước 11: stack trước=[

[1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]

## 9. Cách 3 — Stack lưu `(node, chỉ số con kế tiếp)`: mô phỏng đúng call stack

Cách 2 đẩy **tất cả** con ngay một lúc. Cách 3 bắt chước sát đệ quy hơn: mỗi phần tử stack là cặp `(node, i)` nghĩa là "đang ở node này, con thứ `i` là con kế tiếp cần đi vào". Đây là khuôn mẫu dùng được cho **mọi** kiểu duyệt (pre/post/in-order) – rất đáng học.

- Khi lần đầu đẩy node lên stack → ghi `node.val` (preorder).
- Ở đỉnh `(node, i)`: nếu `i < len(node.children)` → tăng `i` của đỉnh lên, đẩy con thứ `i`. Ngược lại node đã xong → pop.

Ở đây ta dùng list `[node, i]` (có thể sửa được) thay cho tuple (không sửa được).

In [16]:
def preorder_index_stack(root):
    if root is None:
        return []
    result = [root.val]
    stack = [[root, 0]]
    while stack:
        top = stack[-1]                 # xem đỉnh, chưa pop
        node, i = top
        if i < len(node.children):
            top[1] += 1                 # lần sau sẽ xét con kế tiếp
            child = node.children[i]
            result.append(child.val)    # "vào" con -> ghi ngay
            stack.append([child, 0])
        else:
            stack.pop()                 # hết con -> "return" khỏi node
    return result

print(preorder_index_stack(build_nary(EX1)))
print(preorder_index_stack(build_nary(EX2)) == preorder_iterative(build_nary(EX2)))

[1, 3, 5, 6, 2, 4]
True


In [17]:
def trace_index_stack(root):
    result = [root.val]
    stack = [[root, 0]]
    while stack:
        node, i = stack[-1]
        view = [(n.val, k) for n, k in stack]
        if i < len(node.children):
            stack[-1][1] += 1
            child = node.children[i]
            result.append(child.val)
            stack.append([child, 0])
            print(f"stack={view} -> vào con #{i} của {node.val} là {child.val}, result={result}")
        else:
            stack.pop()
            print(f"stack={view} -> {node.val} hết con, pop")
    return result

trace_index_stack(build_nary(EX1))

stack=[(1, 0)] -> vào con #0 của 1 là 3, result=[1, 3]
stack=[(1, 1), (3, 0)] -> vào con #0 của 3 là 5, result=[1, 3, 5]
stack=[(1, 1), (3, 1), (5, 0)] -> 5 hết con, pop
stack=[(1, 1), (3, 1)] -> vào con #1 của 3 là 6, result=[1, 3, 5, 6]
stack=[(1, 1), (3, 2), (6, 0)] -> 6 hết con, pop
stack=[(1, 1), (3, 2)] -> 3 hết con, pop
stack=[(1, 1)] -> vào con #1 của 1 là 2, result=[1, 3, 5, 6, 2]
stack=[(1, 2), (2, 0)] -> 2 hết con, pop
stack=[(1, 2)] -> vào con #2 của 1 là 4, result=[1, 3, 5, 6, 2, 4]
stack=[(1, 3), (4, 0)] -> 4 hết con, pop
stack=[(1, 3)] -> 1 hết con, pop


[1, 3, 5, 6, 2, 4]

## 10. Code nộp LeetCode — bản chính (lặp bằng stack)

In [18]:
class Solution:
    def preorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        stack = [root]
        result = []
        while stack:
            node = stack.pop()
            result.append(node.val)
            if node.children:                       # LeetCode có thể để children = None
                stack.extend(reversed(node.children))
        return result


sol = Solution()
assert sol.preorder(build_nary(EX1)) == [1, 3, 5, 6, 2, 4]
assert sol.preorder(build_nary(EX2)) == [1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]
assert sol.preorder(None) == []
print("Solution (iterative) PASS ví dụ đề")

Solution (iterative) PASS ví dụ đề


## 11. Vì sao đúng? (invariant)

**Invariant của cách stack:** ngay trước mỗi lần `pop`, nối "phần đã ghi trong `result`" với "preorder của các cây con trong stack, lấy theo thứ tự từ **đỉnh xuống đáy**" chính là preorder của cả cây.

- **Ban đầu:** `result = []`, stack chỉ có `root` → đúng là preorder(root).
- **Duy trì:** pop node `x` ở đỉnh. Theo định nghĩa, preorder(x) = `[x] + preorder(c1) + ... + preorder(ck)`. Ta ghi `x` vào `result`, rồi đẩy `ck, ..., c1` để `c1` lên đỉnh → thứ tự đỉnh→đáy bắt đầu bằng `c1, c2, ..., ck` rồi tới các cây con cũ. Invariant được giữ nguyên.
- **Kết thúc:** stack rỗng → `result` chính là preorder của cả cây.

Mỗi node được đẩy vào stack **đúng một lần** (chỉ khi cha của nó bị pop) nên không bị ghi trùng hay bỏ sót.

Với cách đệ quy: chứng minh bằng **quy nạp theo chiều cao** – giả sử `dfs` đúng cho mọi cây con, thì `dfs(node)` ghi `node.val` rồi preorder từng con theo thứ tự → đúng định nghĩa.

## 12. Time / Space Complexity và so sánh các cách

Gọi `n` là số node, `h` là chiều cao cây.

| Cách | Time | Space phụ (không tính output) | Ghi chú |
|---|---|---|---|
| 1. Đệ quy helper | `O(n)` | `O(h)` call stack | Ngắn, dễ hiểu; có thể chạm giới hạn đệ quy |
| 1b. Đệ quy ghép list | `O(n·h)` tệ nhất `O(n²)` | `O(n)` list tạm | Đẹp như công thức nhưng copy nhiều |
| 2. Stack đẩy hết con | `O(n)` | `O(n)` tệ nhất (cây "hình sao": root có n-1 con) | Bản nên nộp cho Follow-up |
| 3. Stack `(node, i)` | `O(n)` | `O(h)` | Mô phỏng đúng đệ quy, dùng được cho postorder |

Vì sao Time `O(n)`: mỗi node được pop đúng 1 lần, và mỗi cạnh cha→con được duyệt đúng 1 lần (tổng số con = `n - 1`).

Lưu ý Space của cách 2: cây "hình sao" root có `n-1` con → stack chứa `n-1` phần tử một lúc.

In [19]:
star = Node(0, [Node(i) for i in range(1, 8)])

def max_stack_size(root):
    stack, best = [root], 1
    while stack:
        node = stack.pop()
        stack.extend(reversed(node.children))
        best = max(best, len(stack))
    return best

print("Cây hình sao 8 node, stack lớn nhất (cách 2):", max_stack_size(star))
print("Cây chuỗi 20000 node, stack lớn nhất (cách 2):", max_stack_size(deep))

Cây hình sao 8 node, stack lớn nhất (cách 2): 7
Cây chuỗi 20000 node, stack lớn nhất (cách 2): 1


## 13. Bộ test: mẫu đề, biên, đối chiếu các cách trên cây ngẫu nhiên

Nhóm test:
1. Hai ví dụ của đề.
2. Biên: cây rỗng, cây 1 node, cây chuỗi, cây hình sao.
3. Đối chiếu: sinh cây ngẫu nhiên (có `seed` để lặp lại được) rồi so 4 cách với nhau.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual if len(str(actual)) < 60 else str(actual)[:57] + '...'}")

methods = {
    "recursive": preorder_recursive,
    "concat": preorder_concat,
    "iterative": preorder_iterative,
    "index_stack": preorder_index_stack,
}

cases = [
    ("đề ví dụ 1", build_nary(EX1), [1, 3, 5, 6, 2, 4]),
    ("đề ví dụ 2", build_nary(EX2), [1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]),
    ("cây rỗng", None, []),
    ("1 node", build_nary([7]), [7]),
    ("hình sao", star, [0, 1, 2, 3, 4, 5, 6, 7]),
    ("chuỗi 6 node", make_chain(6), [0, 1, 2, 3, 4, 5]),
]
for name, root, expected in cases:
    for mname, f in methods.items():
        assert f(root) == expected, (name, mname)
    check(name, preorder_iterative(root), expected)

đề ví dụ 1             PASS -> [1, 3, 5, 6, 2, 4]
đề ví dụ 2             PASS -> [1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]
cây rỗng               PASS -> []
1 node                 PASS -> [7]
hình sao               PASS -> [0, 1, 2, 3, 4, 5, 6, 7]
chuỗi 6 node           PASS -> [0, 1, 2, 3, 4, 5]


In [21]:
def random_nary(n, seed):
    """Sinh cây n node: node mới gắn vào một node cũ chọn ngẫu nhiên."""
    rng = random.Random(seed)
    nodes = [Node(rng.randint(0, 100))]
    for _ in range(n - 1):
        parent = rng.choice(nodes)
        child = Node(rng.randint(0, 100))
        parent.children.append(child)
        nodes.append(child)
    return nodes[0]

for seed in range(200):
    root = random_nary(random.Random(seed).randint(1, 60), seed)
    expected = preorder_recursive(root)
    assert len(expected) > 0
    for mname, f in methods.items():
        assert f(root) == expected, (seed, mname)
print("200 cây ngẫu nhiên: 4 cách cho cùng kết quả — PASS")

200 cây ngẫu nhiên: 4 cách cho cùng kết quả — PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1: Đẩy con theo thứ tự xuôi
```python
stack.extend(node.children)      # SAI: con phải nhất lại ra trước
```
Kết quả bị "lật" thứ tự anh em. Sửa: `stack.extend(reversed(node.children))`.

In [22]:
def preorder_wrong_order(root):
    if root is None:
        return []
    stack, result = [root], []
    while stack:
        node = stack.pop()
        result.append(node.val)
        stack.extend(node.children)      # quên reversed
    return result

print("Sai :", preorder_wrong_order(build_nary(EX1)))
print("Đúng:", preorder_iterative(build_nary(EX1)))

Sai : [1, 4, 2, 3, 6, 5]
Đúng: [1, 3, 5, 6, 2, 4]


### Lỗi 2: Dùng queue (`popleft`) thay cho stack
`popleft` lấy phần tử vào **trước** → thành duyệt theo **tầng** (level order / BFS), không phải preorder.

### Lỗi 3: Quên trường hợp `root is None`
`stack = [None]` rồi `node.val` → `AttributeError`. Luôn kiểm tra cây rỗng trước.

### Lỗi 4: Khai báo `result = []` bên trong `dfs`
Mỗi lần gọi lại tạo list mới → mất kết quả. `result` phải nằm **ngoài** hàm `dfs`.

### Lỗi 5: Dùng tham số mặc định là list
```python
def preorder(root, result=[]):   # SAI: list mặc định được dùng chung giữa các lần gọi!
```

In [23]:
def preorder_bfs_by_mistake(root):
    if root is None:
        return []
    q, result = deque([root]), []
    while q:
        node = q.popleft()
        result.append(node.val)
        q.extend(node.children)
    return result

print("Dùng queue ra level order:", preorder_bfs_by_mistake(build_nary(EX1)))

def preorder_default_list(root, result=[]):
    if root:
        result.append(root.val)
        for c in root.children:
            preorder_default_list(c, result)
    return result

print("Gọi lần 1:", preorder_default_list(build_nary([1, None, 2])))
print("Gọi lần 2:", preorder_default_list(build_nary([1, None, 2])), "<- bị dính kết quả lần 1!")

Dùng queue ra level order: [1, 3, 2, 4, 5, 6]
Gọi lần 1: [1, 2]
Gọi lần 2: [1, 2, 1, 2] <- bị dính kết quả lần 1!


## 15. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:** đề nói "traversal", "preorder", "thăm node trước con", "DFS", cây N-ary / nhị phân, Follow-up "iterative".

**Pattern DFS bằng stack:**
```text
stack = [root]
while stack:
    node = stack.pop()
    xử lý node
    đẩy con theo thứ tự NGƯỢC thứ tự muốn thăm
```

**Biến thể thường gặp:**
- **590. N-ary Tree Postorder** – ghi cha sau con (mẹo: preorder kiểu "phải→trái" rồi đảo ngược).
- **144. Binary Tree Preorder** – y hệt, chỉ thay `children` bằng `[left, right]`.
- **429. N-ary Tree Level Order** – dùng queue thay stack.
- **559. Maximum Depth of N-ary Tree** – DFS mang theo độ sâu.

Demo nhanh: dùng chung code cho cây nhị phân bằng cách coi `children = [left, right]` (bỏ `None`).

In [24]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def binary_preorder(root):
    if root is None:
        return []
    stack, result = [root], []
    while stack:
        node = stack.pop()
        result.append(node.val)
        for child in (node.right, node.left):   # đẩy phải trước, trái sau
            if child:
                stack.append(child)
    return result

#      1
#     / \
#    2   3
#   /
#  4
t = TreeNode(1, TreeNode(2, TreeNode(4)), TreeNode(3))
print(binary_preorder(t))   # [1, 2, 4, 3]

[1, 2, 4, 3]


## 16. Bài tập (tự làm, KHÔNG chạy cell TODO khi chưa xong)

**Bài 1 — Tính tay.** Cho `[10,null,20,30,null,40,50,null,60]`. Vẽ cây và viết preorder. (Kiểm bằng code ở phần đáp án.)

**Bài 2 — Đếm số node** của cây N-ary bằng đệ quy.

**Bài 3 — Viết lại preorder bằng stack** (không nhìn code phía trên).

**Bài 4 — Độ sâu tối đa** (LeetCode 559): số node trên đường dài nhất từ root tới lá.

**Bài 5 — Level order** (LeetCode 429): trả về list các tầng, mỗi tầng là list giá trị.

**Bài 6 — Mini challenge:** trả về **tất cả đường đi root → lá** dạng chuỗi `"1->3->5"`, theo thứ tự preorder, bằng stack (không đệ quy).

In [25]:
# Bài 1: tự vẽ cây trên giấy rồi điền vào đây
answer_ex1_by_hand = None   # TODO: thay None bằng list preorder bạn tính được


def count_nodes(root):
    # TODO: base case None -> 0; còn lại 1 + tổng count_nodes của các con
    pass


def preorder_stack_exercise(root):
    # TODO 1: cây rỗng -> []
    # TODO 2: stack = [root], result = []
    # TODO 3: pop, ghi, đẩy con theo thứ tự ngược
    pass

# Kiểm tra sau khi làm:
# assert count_nodes(build_nary(EX2)) == 14
# assert preorder_stack_exercise(build_nary(EX1)) == [1, 3, 5, 6, 2, 4]

In [26]:
def max_depth(root):
    # TODO: cây rỗng -> 0; ngược lại 1 + max độ sâu các con (0 nếu không có con)
    pass


def level_order(root):
    # TODO: dùng deque; mỗi vòng while xử lý trọn một tầng (len(queue) phần tử)
    pass


def all_paths(root):
    # TODO: stack chứa cặp (node, path_string); gặp lá thì ghi path
    pass

# Kiểm tra sau khi làm:
# assert max_depth(build_nary(EX1)) == 3
# assert level_order(build_nary(EX1)) == [[1], [3, 2, 4], [5, 6]]
# assert all_paths(build_nary(EX1)) == ["1->3->5", "1->3->6", "1->2", "1->4"]

## 17. Gợi ý 3 mức

**Bài 2 – đếm node**
1. Hướng: số node của cây = 1 (root) + số node của từng cây con.
2. Cú pháp: `sum(count_nodes(c) for c in root.children)`.
3. Gần code: `return 0 if root is None else 1 + sum(...)`.

**Bài 3 – stack**
1. Hướng: stack chứa các cây con còn chờ; đỉnh là cây con sẽ duyệt kế tiếp.
2. Cú pháp: `stack.pop()`, `stack.extend(reversed(...))`.
3. Gần code: xem bảng trace mục 8 và tự viết lại từ bảng.

**Bài 4 – độ sâu**
1. Hướng: độ sâu = 1 + độ sâu lớn nhất trong các con.
2. Cú pháp: `max(..., default=0)` để xử lý node lá (list rỗng).
3. Gần code: `1 + max((max_depth(c) for c in root.children), default=0)`.

**Bài 5 – level order**
1. Hướng: queue (FIFO); trước mỗi tầng ghi `size = len(queue)`.
2. Cú pháp: `for _ in range(size): node = queue.popleft()`.
3. Gần code: sau khi lấy node, `queue.extend(node.children)` và gom `node.val` vào `level`.

**Bài 6 – đường đi**
1. Hướng: mang theo "đường đi tới node" cùng node trên stack.
2. Cú pháp: `stack.append((child, path + "->" + str(child.val)))`.
3. Gần code: node không có con → `paths.append(path)`; nhớ đẩy con theo thứ tự ngược để kết quả theo preorder.

## 18. Đáp án tham khảo, template nhớ nhanh và checklist submit

### Đáp án (chạy kiểm tra)

In [27]:
def ans_count_nodes(root):
    if root is None:
        return 0
    return 1 + sum(ans_count_nodes(c) for c in root.children)


def ans_preorder_stack(root):
    if root is None:
        return []
    stack, result = [root], []
    while stack:
        node = stack.pop()
        result.append(node.val)
        stack.extend(reversed(node.children))
    return result


def ans_max_depth(root):
    if root is None:
        return 0
    return 1 + max((ans_max_depth(c) for c in root.children), default=0)


def ans_level_order(root):
    if root is None:
        return []
    queue, levels = deque([root]), []
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            queue.extend(node.children)
        levels.append(level)
    return levels


def ans_all_paths(root):
    if root is None:
        return []
    stack, paths = [(root, str(root.val))], []
    while stack:
        node, path = stack.pop()
        if not node.children:
            paths.append(path)
        for child in reversed(node.children):
            stack.append((child, path + "->" + str(child.val)))
    return paths


# Bài 1: cây [10,null,20,30,null,40,50,null,60]
ex_tree = build_nary([10, None, 20, 30, None, 40, 50, None, 60])
show_nary(ex_tree)
print("Preorder bài 1:", ans_preorder_stack(ex_tree))   # [10, 20, 40, 50, 30, 60]

assert ans_preorder_stack(ex_tree) == [10, 20, 40, 50, 30, 60]
assert ans_count_nodes(build_nary(EX2)) == 14
assert ans_count_nodes(None) == 0
assert ans_preorder_stack(build_nary(EX1)) == [1, 3, 5, 6, 2, 4]
assert ans_max_depth(build_nary(EX1)) == 3
assert ans_max_depth(build_nary(EX2)) == 5
assert ans_level_order(build_nary(EX1)) == [[1], [3, 2, 4], [5, 6]]
assert ans_all_paths(build_nary(EX1)) == ["1->3->5", "1->3->6", "1->2", "1->4"]
print("Đáp án: tất cả test PASS")

10
    20
        40
        50
    30
        60
Preorder bài 1: [10, 20, 40, 50, 30, 60]
Đáp án: tất cả test PASS


### Template nhớ nhanh

```python
class Solution:
    def preorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        stack, result = [root], []
        while stack:
            node = stack.pop()
            result.append(node.val)                 # ghi TRƯỚC (pre)
            if node.children:
                stack.extend(reversed(node.children))  # đẩy NGƯỢC để con trái ra trước
        return result
```

Câu thần chú: **"pop – ghi – đẩy ngược"**.

### Checklist trước khi submit
- [ ] Xử lý `root is None` → `[]`?
- [ ] Ghi `node.val` **trước** khi xử lý con?
- [ ] Đẩy con bằng `reversed(...)` (nếu dùng stack)?
- [ ] Dùng `stack.pop()` (cuối list), **không** phải `pop(0)`/`popleft()`?
- [ ] Với bản đệ quy: `result` khai báo ngoài `dfs`, không dùng list làm tham số mặc định?
- [ ] Đã thử cây rỗng, 1 node, cây chuỗi, cây hình sao?